In [1]:
!pip install vllm==0.10.2 python-mecab-ko rouge

INFO: pip is looking at multiple versions of transformers to determine which version is compatible with other requirements. This could take a while.
INFO: pip is still looking at multiple versions of transformers to determine which version is compatible with other requirements. This could take a while.
INFO: This is taking longer than usual. You might need to provide the dependency resolver with stricter constraints to reduce runtime. See https://pip.pypa.io/warnings/backtracking for guidance. If you want to abort this run, press Ctrl + C.
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 436.4/436.4 MB ?  0:09:29eta 0:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.9/3.9 MB 75.8 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.3/2.3 MB 68.7 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 117.2/117.2 MB 105.2 MB/s  0:00:010:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.9/7.9 MB 86.1 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 15.0/15.0 M

In [2]:
import json, ast
import pandas as pd
from transformers import AutoTokenizer
from vllm import LLM, SamplingParams
from vllm.lora.request import LoRARequest
from mecab import MeCab
from rouge import Rouge

INFO 07-13 00:51:12 [__init__.py:216] Automatically detected platform cuda.


In [3]:
# 학습 때 저장한 held-out 평가셋을 그대로 불러온다 (학습에 쓰이지 않은 데이터)
df_eval = pd.read_json("eval_split.jsonl", lines=True, encoding="utf-8")

tokenizer = AutoTokenizer.from_pretrained("Qwen/Qwen3-1.7B")

system_prompt = """당신은 뉴스 텍스트를 분석하여 카테고리를 분류하고 주요 핵심 이벤트들을 추출하는 전문 분석 시스템입니다.
주어진 텍스트를 분석하여 반드시 파이썬의 dictionary 형식으로 결과를 작성하십시오.
큰 따옴표 사이에 다른 따옴표들을 적으려고 시도하지 마십시오. 이는 dictionary 파싱을 실패하게 하는 원인이 됩니다.
아래 dictionary에서 각 value는 지시사항에 해당합니다. 지시사항을 그대로 적지 마시고, 해당 지시사항에 따라 적절한 value를 채워넣으십시오.

분석 결과는 다음 형식으로 작성하십시오:

답변:
{"category": "텍스트의 카테고리를 ['부동산', '산업', '오피니언', '증권'] 중 하나로 분류하여 작성하십시오",
"event_count": "텍스트 내에서 발견된 핵심 이벤트의 개수를 정수로 작성하십시오(증감, 변화, 변동이 나오는 구절을 중점적으로 보세요.)",
"events": ["텍스트에서 추출한 각 핵심 이벤트를 요약하는 문장들을 파이썬 문자열 리스트 형태로 작성하십시오. 원문의 핵심 내용을 그대로 유지하되 한 문장으로 간결하게 작성하십시오"]}"""

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json:   0%|          | 0.00/11.4M [00:00<?, ?B/s]

In [4]:
prompt_lst, label_lst = [], []
for _, row in df_eval.iterrows():
    messages = [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": f"다음 뉴스 텍스트를 분석해주세요:\n\n{row['text']}"},
    ]
    prompt = tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True, enable_thinking=False
    )
    prompt_lst.append(prompt)

    events = [str(row[f"event_sentence{i}"]).strip()
              for i in range(1, 11) if pd.notna(row[f"event_sentence{i}"])]
    label_lst.append({
        "category": row["text_category"],
        "event_count": int(row["event_quantity"]),
        "events": events,
    })

In [5]:
sampling_params = SamplingParams(temperature=0.0, max_tokens=1024, stop=["<|im_end|>"])

llm = LLM(model="Qwen/Qwen3-1.7B", enable_lora=True, max_lora_rank=16, max_model_len=8192)

# (1) 베이스: 어댑터 없이
base_preds = [o.outputs[0].text for o in llm.generate(prompt_lst, sampling_params)]

# (2) 파인튜닝: 학습한 LoRA 어댑터를 붙여서
lora_request = LoRARequest("qwen3-ft", 1, "./qwen3-lora-ft-final")
lora_preds = [o.outputs[0].text for o in llm.generate(prompt_lst, sampling_params, lora_request=lora_request)]

INFO 07-13 00:51:17 [utils.py:328] non-default args: {'max_model_len': 8192, 'disable_log_stats': True, 'enable_lora': True, 'model': 'Qwen/Qwen3-1.7B'}


config.json:   0%|          | 0.00/726 [00:00<?, ?B/s]

INFO 07-13 00:51:27 [__init__.py:742] Resolved architecture: Qwen3ForCausalLM


`torch_dtype` is deprecated! Use `dtype` instead!


INFO 07-13 00:51:27 [__init__.py:1815] Using max model len 8192
INFO 07-13 00:51:29 [scheduler.py:222] Chunked prefill is enabled with max_num_batched_tokens=8192.
WARNING 07-13 00:51:29 [lora.py:92] `lora_extra_vocab_size` is deprecated and will be removed in v0.12.0. Additional vocabulary support for LoRA adapters is being phased out.


...

(EngineCore_DP0 pid=2363) INFO 07-13 00:53:09 [gpu_model_runner.py:3118] Graph capturing finished in 25 secs, took 0.60 GiB
(EngineCore_DP0 pid=2363) INFO 07-13 00:53:09 [gpu_worker.py:391] Free memory on device (78.76/79.25 GiB) on startup. Desired GPU memory utilization is (0.9, 71.32 GiB). Actual usage is 3.25 GiB for weight, 1.42 GiB for peak activation, 0.02 GiB for non-torch memory, and 0.6 GiB for CUDAGraph memory. Replace gpu_memory_utilization config with `--kv-cache-memory=70749399859` to fit into requested memory, or `--kv-cache-memory=78733637632` to fully utilize gpu memory. Current kv cache memory in use is 71552609075 bytes.
(EngineCore_DP0 pid=2363) INFO 07-13 00:53:20 [core.py:218] init engine (profile, create kv cache, warmup model) took 95.49 seconds
INFO 07-13 00:53:21 [llm.py:295] Supported_tasks: ['generate']
INFO 07-13 00:53:21 [__init__.py:36] No IOProcessor plugins requested by the model


Adding requests:   0%|          | 0/326 [00:00<?, ?it/s]

Processed prompts:   0%|          | 0/326 [00:00<?, ?it/s, est. speed input: 0.00 toks/s, output: 0.00 toks/s]

Adding requests:   0%|          | 0/326 [00:00<?, ?it/s]

Processed prompts:   0%|          | 0/326 [00:00<?, ?it/s, est. speed input: 0.00 toks/s, output: 0.00 toks/s]

In [6]:
mecab = MeCab()
rouge = Rouge()

def parse_dict(text):
    fragment = text[text.find("{"): text.rfind("}") + 1]
    try:
        return json.loads(fragment)
    except Exception:
        try:
            return ast.literal_eval(fragment)
        except Exception:
            return None

def rouge_f(ref_events, hyp_events):
    ref = " ".join(mecab.morphs(" ".join(ref_events)))
    hyp = " ".join(mecab.morphs(" ".join(hyp_events)))
    if not ref.strip() or not hyp.strip():
        return 0.0, 0.0, 0.0
    s = rouge.get_scores(hyp, ref)[0]      # (예측, 정답) 순서
    return s["rouge-1"]["f"], s["rouge-2"]["f"], s["rouge-l"]["f"]

In [7]:
def evaluate(preds, labels):
    rows = []
    for pred, label in zip(preds, labels):
        p = parse_dict(pred)
        if p is None:
            rows.append({"parse_ok": 0.0, "category_acc": 0.0, "count_acc": 0.0,
                         "rouge1": 0.0, "rouge2": 0.0, "rougeL": 0.0})
            continue
        pev = p.get("events") if isinstance(p.get("events"), list) else []
        r1, r2, rl = rouge_f(label["events"], pev)
        rows.append({
            "parse_ok":     1.0,
            "category_acc": 1.0 if p.get("category")    == label["category"]    else 0.0,
            "count_acc":    1.0 if p.get("event_count") == label["event_count"] else 0.0,
            "rouge1": r1, "rouge2": r2, "rougeL": rl,
        })
    return pd.DataFrame(rows)

base_df = evaluate(base_preds, label_lst)
lora_df = evaluate(lora_preds, label_lst)

In [8]:
comp = pd.DataFrame({"기존(베이스)": base_df.mean(), "신규(파인튜닝)": lora_df.mean()})
comp["차이"] = comp["신규(파인튜닝)"] - comp["기존(베이스)"]

base_df.to_csv("base_eval.csv", index=False, encoding="utf-8-sig")
lora_df.to_csv("lora_eval.csv", index=False, encoding="utf-8-sig")
comp.to_csv("comparison.csv", encoding="utf-8-sig")
print(comp.round(4))

              기존(베이스)  신규(파인튜닝)      차이
parse_ok       0.7945    0.9632  0.1687
category_acc   0.0798    0.8926  0.8129
count_acc      0.0552    0.6442  0.5890
rouge1         0.3122    0.6119  0.2997
rouge2         0.1905    0.5120  0.3215
rougeL         0.2946    0.6001  0.3056
